In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.functions import to_date, date_format
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### ## Clean DimChannel

In [0]:
channel_df = spark.read.format("parquet")\
         .option("inferSchema", "true")\
         .option("header", "true")\
         .load("abfss://tvbronze@saazurede.dfs.core.windows.net/DimChannel")

target_table = "tv_cata.silver.DimChannel"

# Check if the table already exists
if spark.catalog.tableExists(target_table):
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        channel_df.alias("s"),
        "t.channel_id = s.channel_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    # First run — table doesn't exist yet, so just create it
    channel_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)
# demo_df = demo_df.drop("hh_child_key", "hh_sec_key", "indi_income_key", "indi_education_key", "indi_occupancy_key","hh_income_key", "hh_education_key", "hh_size_key","mother_with_child")
# demo_df.display()

### ## Clean DemoTable

In [0]:
demo_df = spark.read.format("parquet")\
         .option("inferSchema", "true")\
         .option("header", "true")\
         .load("abfss://tvbronze@saazurede.dfs.core.windows.net/b_d_raw")
    
demo_df = demo_df.drop("hh_child_key", "hh_sec_key", "indi_income_key", "indi_education_key", "indi_occupancy_key","hh_income_key", "hh_education_key", "hh_size_key","mother_with_child")

target_table = "tv_cata.silver.fctdemo"

# Check if the table already exists
if spark.catalog.tableExists(target_table):
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        demo_df.alias("s"),
        "t.memberid = s.memberid AND t.date = s.date"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    # First run — table doesn't exist yet, so just create it
    demo_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

### ## Clean StatementTable

In [0]:
statment_df = spark.read.format("parquet")\
         .option("inferSchema", "true")\
         .option("header", "true")\
         .load("abfss://tvbronze@saazurede.dfs.core.windows.net/b_s_raw")

statment_df = statment_df.withColumn("end",col("start")+col("duration"))
target_table = "tv_cata.silver.fctstatement"

# Check if the table already exists
if spark.catalog.tableExists(target_table):
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        statment_df.alias("s"),
        "t.memberid = s.memberid AND t.date = s.date and t.channel_id = s.channel_id and t.start = s.start"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    # First run — table doesn't exist yet, so just create it
    statment_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

playout_df = spark.read.format("parquet")\
         .option("inferSchema", "true")\
         .option("header", "true")\
         .load("abfss://tvbronze@saazurede.dfs.core.windows.net/b_p_raw")

# Dedupe: keep only one row per (channel_id, event_date, time_from)
window = Window.partitionBy("channel_id", "event_date", "time_from").orderBy(col("Updated_at").desc())

playout_df = (playout_df
    .withColumn("rn", row_number().over(window))
    .filter("rn = 1")
    .drop("rn"))

target_table = "tv_cata.silver.fctplayout"

if spark.catalog.tableExists(target_table):
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        playout_df.alias("s"),
        "t.channel_id = s.channel_id AND t.event_date = s.event_date AND t.time_from = s.time_from"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    playout_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

### Processed s_sd_processed table

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col, to_date

statment_df = statment_df.drop("Updated_at")
demo_df = demo_df.drop("Updated_at")

s_sd_processed_df = statment_df.join(
    demo_df,
    (statment_df.memberid == demo_df.memberid) & (statment_df.date == demo_df.date),
    "inner"
).select(
    statment_df.memberid.alias("member_id"),
    statment_df.start.alias("start"),
    statment_df.end.alias("end"),
    statment_df.duration.alias("duration"),
    statment_df.channel_id.alias("channel_id"),
    statment_df.date.alias("event_date"),
    demo_df.citygroup.alias("citygroup"),
    demo_df.gender,
    demo_df.agegroup,
    demo_df.actual_age,
    demo_df.weight
)

s_sd_processed_df = s_sd_processed_df.withColumn("event_date", to_date(col("event_date")))

# Dedupe on the full natural key before merging
window = Window.partitionBy("member_id", "channel_id", "event_date", "start").orderBy(col("duration").desc())
s_sd_processed_df = (s_sd_processed_df
    .withColumn("rn", row_number().over(window))
    .filter("rn = 1")
    .drop("rn"))

target_table = "tv_cata.silver.fct_sd_processed"

if spark.catalog.tableExists(target_table):
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        s_sd_processed_df.alias("s"),
        "t.channel_id = s.channel_id AND t.event_date = s.event_date AND t.start = s.start AND t.member_id = s.member_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    s_sd_processed_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)

### Update S_dim_30min

In [0]:
# Step 1: Get min and max date from your existing df
min_max_df = s_sd_processed_df.select(
    min("event_date").alias("min_date"),
    max("event_date").alias("max_date")
)

# Step 2: Generate the full sequence of dates between them
date_df = min_max_df.select(
    explode(
        sequence(col("min_date"), col("max_date"), expr("interval 1 day"))
    ).alias("event_date")
)

channel_df = channel_df.select("channel_id", "channel_name").distinct()


def format_time(total_seconds):
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"

# Start at 02:00 (7200 seconds from midnight), 48 bands of 30 mins = full 24 hours
start_of_day_sec = 2 * 3600   # 7200 -> 02:00
interval_sec = 30 * 60        # 1800 seconds = 30 minutes
num_bands = 48                 # 24 hours / 30 min

rows = []
for i in range(num_bands):
    start_sec = start_of_day_sec + (i * interval_sec)
    end_sec = start_sec + interval_sec
    
    time_band_start = format_time(start_sec)
    time_band_end = format_time(end_sec)
    time_band = f"{time_band_start} - {time_band_end}"
    
    rows.append((time_band_start, time_band_end, start_sec, end_sec, time_band))

schema = StructType([
    StructField("time_band_start", StringType(), False),
    StructField("time_band_end", StringType(), False),
    StructField("start_sec", IntegerType(), False),
    StructField("end_sec", IntegerType(), False),
    StructField("time_band", StringType(), False),
])

timeband_df = spark.createDataFrame(rows, schema)

s_dim_30min_df = channel_df.crossJoin(date_df).crossJoin(timeband_df)

target_table = "tv_cata.silver.dim_Date_channel_30min"
# Check if the table already exists
if spark.catalog.tableExists(target_table):
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        s_dim_30min_df.alias("s"),
        "t.channel_id = s.channel_id AND t.event_date = s.event_date and t.start_sec = s.start_sec "
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll()\
     .execute()

else:
    # First run — table doesn't exist
        s_dim_30min_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(target_table)
